In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import calendar

# Load dataset
file_path = r"C:\Users\Asus\Desktop\Border_Crossing_Entry_Data.csv"
df = pd.read_csv(file_path, low_memory=False)

# Data Preprocessing
df = df.dropna()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')  # General auto-detection
df = df.dropna(subset=['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name()

# Ensure months are in the correct order
month_order = list(calendar.month_name)[1:]  # Jan to Dec

# 1. Monthly and Seasonal Trends
monthly_trend = df.groupby(['Year', 'Month_Name'])['Value'].sum().unstack()
monthly_trend = monthly_trend[month_order]  # Ensure proper order

print("Monthly Trend Data:\n", monthly_trend)
plt.figure(figsize=(12, 6))
sns.heatmap(monthly_trend, cmap='coolwarm', annot=True, fmt='.0f')
plt.title("Monthly Border Crossing Trends")
plt.ylabel("Year")
plt.xlabel("Month")
plt.show()

# 2. Border and State-Wise Traffic Comparison
border_traffic = df.groupby('Border')['Value'].sum()
print("Total Traffic by Border:\n", border_traffic)
plt.figure(figsize=(8, 5))
sns.barplot(x=border_traffic.index, y=border_traffic.values, palette="viridis")
plt.title("Traffic Comparison: US-Canada vs. US-Mexico Borders")
plt.ylabel("Total Crossings")
plt.show()

state_traffic = df.groupby('State')['Value'].sum().sort_values(ascending=False)
print("Top 10 States by Border Traffic:\n", state_traffic.head(10))
plt.figure(figsize=(12, 6))
sns.barplot(x=state_traffic.head(10).index, y=state_traffic.head(10).values, palette="magma")
plt.xticks(rotation=45)
plt.title("Top 10 States by Border Traffic")
plt.ylabel("Total Crossings")
plt.show()

# 3. Top 5 Busiest and Least Busy Ports
port_traffic = df.groupby('Port Name')['Value'].sum().sort_values(ascending=False)
print("Top 5 Busiest Ports:\n", port_traffic.head(5))
plt.figure(figsize=(12, 6))
sns.barplot(x=port_traffic.head(5).index, y=port_traffic.head(5).values, palette="coolwarm")
plt.xticks(rotation=45)
plt.title("Top 5 Busiest Ports")
plt.ylabel("Total Crossings")
plt.show()

print("Top 5 Least Busy Ports:\n", port_traffic.tail(5))
plt.figure(figsize=(12, 6))
sns.barplot(x=port_traffic.tail(5).index, y=port_traffic.tail(5).values, palette="Blues")
plt.xticks(rotation=45)
plt.title("Least Busy Ports")
plt.ylabel("Total Crossings")
plt.show()

# 4. Mode of Transport Analysis
transport_mode = df.groupby('Measure')['Value'].sum().sort_values(ascending=False)
print("Traffic by Mode of Transport:\n", transport_mode)
plt.figure(figsize=(12, 6))
sns.barplot(x=transport_mode.index, y=transport_mode.values, palette="crest")
plt.xticks(rotation=45)
plt.title("Border Crossings by Mode of Transport")
plt.ylabel("Total Crossings")
plt.show()

# 5. Geospatial Visualization
# Only attempt this if longitude/latitude data is available
if 'Longitude' in df.columns and 'Latitude' in df.columns:
    df = df.dropna(subset=['Longitude', 'Latitude'])
    df['geometry'] = df.apply(lambda row: Point(row['Longitude'], row['Latitude']), axis=1)
    gdf = gpd.GeoDataFrame(df, geometry='geometry')

    print("Border Crossing Geospatial Data (Sample):\n", gdf.head())

    # Use a local copy of Natural Earth if needed or download manually
    try:
        world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
        fig, ax = plt.subplots(figsize=(10, 6))
        world.boundary.plot(ax=ax, color='black')
        gdf.plot(ax=ax, markersize=10, alpha=0.5, color='red')
        plt.title("Border Crossing Locations")
        plt.show()
    except Exception as e:
        print("Geospatial data loading failed:", e)
else:
    print("Longitude and Latitude columns missing! Skipping geospatial visualization.")

# 6. Year-over-Year Growth Analysis
yearly_growth = df.groupby('Year')['Value'].sum()
print("Yearly Growth Data:\n", yearly_growth)
plt.figure(figsize=(10, 5))
sns.lineplot(x=yearly_growth.index, y=yearly_growth.values, marker='o', color='brown')
plt.title("Yearly Border Traffic Growth")
plt.ylabel("Total Crossings")
plt.xlabel("Year")
plt.grid()
plt.show()

print("Analysis Complete!")
